# 01B · Home vs Away overview

Análisis del dashboard de equipos enfocado en el split de localía (Regular Season 2024-25).

## 1. Introducción

En este cuaderno exploramos cómo varían las métricas de rendimiento de cada franquicia dependiendo de si juegan en casa (`Home`) o fuera (`Away`). Evaluaremos resultados, eficiencia ofensiva y correlaciones clave para detectar qué factores impulsan el éxito según la localización.

## 2. Carga de datos

Leemos el parquet consolidado de `team_dashboard_by_general_splits`, filtrando la temporada objetivo y seleccionando únicamente el `source_dataset = 1`, que corresponde a los splits Home/Away. Se inspecciona la forma del DataFrame, el listado de columnas disponibles y las primeras filas para validar el filtrado.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[3]
DATA_DIR = PROJECT_ROOT / "00_data"
FIGURES_DIR = PROJECT_ROOT / "02a_reports" / "figures" / "home_away"
TABLES_DIR = PROJECT_ROOT / "02a_reports" / "tables" / "home_away"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

INTERMEDIATE_PATH = (
    DATA_DIR
    / "00b_intermediate"
    / "team_dashboard"
    / "general_splits"
    / "2024-25"
    / "Regular Season"
    / "team_dashboard__general_splits.parquet"
)
DATASET_REFERENCE_ID = 0
SOURCE_DATASET_ID = 1
SEASON_TARGET = "2024-25"

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Parquet intermedio: {INTERMEDIATE_PATH}")
print(f"Dataset de referencia: {DATASET_REFERENCE_ID}")
print(f"Source dataset (Home/Away): {SOURCE_DATASET_ID}")
print(f"Temporada objetivo: {SEASON_TARGET}")

In [ ]:
general_df = pd.read_parquet(INTERMEDIATE_PATH)

colmap = {c.lower(): c for c in general_df.columns}

df = general_df.copy()

ds_col = colmap.get("dataset")
if ds_col:
    df = df[df[ds_col].astype(str) == str(DATASET_REFERENCE_ID)]
else:
    print("⚠️ No se encontró columna 'dataset'. Se mantiene el DataFrame original.")

sds_col = colmap.get("source_dataset")
if sds_col:
    df = df[df[sds_col].astype(str) == str(SOURCE_DATASET_ID)]
else:
    print("⚠️ No se encontró columna 'source_dataset'.")

season_col = colmap.get("season") or colmap.get("season_year")
if season_col:
    df = df[df[season_col].astype(str) == str(SEASON_TARGET)]
else:
    print("⚠️ No se encontró columna de temporada ('season' o 'season_year').")

analysis_df = df.copy()

print("general_df shape:", general_df.shape)
print("analysis_df shape:", analysis_df.shape)
print("columnas disponibles:", list(analysis_df.columns))
if sds_col:
    print("source_dataset únicos:", sorted(analysis_df[sds_col].astype(str).unique()))
if season_col:
    print("season únicos:", sorted(analysis_df[season_col].astype(str).unique()))

display(analysis_df.head())

## 3. Resumen general Home vs Away

Agregamos la información por franquicia y localización para resumir victorias, derrotas y porcentaje de victorias (`W_PCT`). También calculamos el total de partidos jugados combinando ambos escenarios.

In [ ]:
analysis_std = analysis_df.rename(columns={col: col.upper() for col in analysis_df.columns})
COLUMN_LOOKUP = {col: col for col in analysis_std.columns}

def require_columns(labels):
    missing = [label for label in labels if label not in COLUMN_LOOKUP]
    if missing:
        raise KeyError(f"Columnas faltantes en el parquet filtrado: {missing}")
    return [COLUMN_LOOKUP[label] for label in labels]

TEAM_KEYS = require_columns(["TEAM_ID", "TEAM_NAME"])
location_col = require_columns(["GROUP_VALUE"])[0]

summary_needed = ["GP", "W", "L", "W_PCT"]
available_summary = [c for c in summary_needed if c in COLUMN_LOOKUP]

if not {"W", "L"}.issubset(set(available_summary)):
    print("⚠️ No hay columnas suficientes de victorias/derrotas para construir el resumen Home/Away.")
    summary_table = analysis_std[TEAM_KEYS].drop_duplicates().reset_index(drop=True)
else:
    summary_cols = require_columns(available_summary)

    summary_df = analysis_std[TEAM_KEYS + [location_col] + summary_cols].copy()

    def normalize_location(value):
        text = str(value).strip().lower()
        if "home" in text:
            return "HOME"
        if "away" in text or "road" in text:
            return "AWAY"
        return text.upper() if text else "UNKNOWN"

    summary_df["LOCATION"] = summary_df[location_col].map(normalize_location)

    agg_funcs = {col: "sum" for col in summary_cols}
    location_summary = (
        summary_df.groupby(TEAM_KEYS + ["LOCATION"], dropna=False)
        .agg(agg_funcs)
        .reset_index()
    )

    from functools import reduce
    pivot_frames = []
    for metric in available_summary:
        pivot = (
            location_summary
            .pivot_table(index=TEAM_KEYS, columns="LOCATION", values=metric, aggfunc="sum")
            .rename(columns=lambda loc: f"{metric}_{loc}")
        )
        pivot_frames.append(pivot)

    summary_wide = reduce(lambda left, right: left.join(right, how="outer"), pivot_frames)
    summary_wide = summary_wide.reset_index()

    if {f"GP_{loc}" for loc in ["HOME", "AWAY"]}.issubset(summary_wide.columns):
        summary_wide["TOTAL_GP"] = summary_wide.get("GP_HOME", 0).fillna(0) + summary_wide.get("GP_AWAY", 0).fillna(0)
    else:
        summary_wide["TOTAL_GP"] = summary_wide.filter(like="GP_").sum(axis=1)

    ordered_cols = TEAM_KEYS + [
        "W_HOME", "L_HOME", "W_PCT_HOME",
        "W_AWAY", "L_AWAY", "W_PCT_AWAY",
        "TOTAL_GP",
    ]
    existing_cols = [col for col in ordered_cols if col in summary_wide.columns]
    summary_table = summary_wide[existing_cols + [col for col in summary_wide.columns if col not in existing_cols]]

if not summary_table.empty:
    display(summary_table.head())

summary_path = TABLES_DIR / "resumen_home_away.csv"
summary_table.to_csv(summary_path, index=False)
print("Resumen exportado a", summary_path)

## 4. Métricas ofensivas Home vs Away

Comparamos porcentajes de tiro, puntos y diferencial (`PLUS_MINUS`) para cada franquicia según juegue en casa o fuera. Se generan gráficos de barras agrupadas y se exportan las figuras.

In [ ]:
offensive_metrics = ["FG_PCT", "FG3_PCT", "FT_PCT", "PTS", "PLUS_MINUS"]
available_offensive = [metric for metric in offensive_metrics if metric in COLUMN_LOOKUP]

if not available_offensive:
    print("⚠️ No hay métricas ofensivas disponibles para el análisis Home/Away.")
    offensive_table = analysis_std[TEAM_KEYS].drop_duplicates().reset_index(drop=True)
else:
    off_cols = require_columns(available_offensive)

    off_df = analysis_std[TEAM_KEYS + [location_col] + off_cols].copy()
    off_df["LOCATION"] = off_df[location_col].map(normalize_location)

    agg_off = (
        off_df.groupby(TEAM_KEYS + ["LOCATION"], dropna=False)
        .agg({col: "mean" for col in off_cols})
        .reset_index()
    )

    from functools import reduce
    wide_metrics = []
    for metric in available_offensive:
        pivot = (
            agg_off.pivot_table(index=TEAM_KEYS, columns="LOCATION", values=metric, aggfunc="mean")
            .rename(columns=lambda loc: f"{metric}_{loc}")
        )
        wide_metrics.append(pivot)

    offensive_table = reduce(lambda left, right: left.join(right, how="outer"), wide_metrics).reset_index()

    table_off_path = TABLES_DIR / "ofensiva_home_away.csv"
    offensive_table.to_csv(table_off_path, index=False)
    print("Tabla ofensiva exportada a", table_off_path)

    display(offensive_table.head())

    for metric in available_offensive:
        metric_cols = [f"{metric}_{loc}" for loc in ["HOME", "AWAY"] if f"{metric}_{loc}" in offensive_table.columns]
        if len(metric_cols) < 2:
            print(f"⚠️ Métrica '{metric}' no tiene ambos splits. Se omite el gráfico.")
            continue

        plot_df = offensive_table[[COLUMN_LOOKUP["TEAM_NAME"]] + metric_cols].copy()
        plot_df = plot_df.sort_values(metric_cols[0], ascending=False)

        fig, ax = plt.subplots(figsize=(14, 6))
        width = 0.35
        x = range(len(plot_df))
        ax.bar([i - width / 2 for i in x], plot_df[metric_cols[0]], width=width, label=metric_cols[0].split('_')[-1].title())
        ax.bar([i + width / 2 for i in x], plot_df[metric_cols[1]], width=width, label=metric_cols[1].split('_')[-1].title())
        ax.set_xticks(list(x))
        ax.set_xticklabels(plot_df[COLUMN_LOOKUP["TEAM_NAME"]], rotation=45, ha="right")
        ax.set_title(f"{metric}: Home vs Away")
        ax.set_xlabel("Equipo")
        ax.set_ylabel(metric)
        if metric.endswith("_PCT"):
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
        ax.legend()
        plt.tight_layout()

        figure_path = FIGURES_DIR / f"{metric.lower()}_home_vs_away.png"
        fig.savefig(figure_path, dpi=150)
        plt.show()
        print("Figura guardada en", figure_path)

## 5. Correlaciones

Creamos una matriz de correlación que considera las métricas ofensivas clave duplicadas por localización. Esto permite detectar patrones que se repiten en casa y fuera.

In [ ]:
corr_metrics = ["W_PCT", "FG_PCT", "FG3_PCT", "FT_PCT", "AST", "TOV", "PTS", "PLUS_MINUS"]
available_corr = [metric for metric in corr_metrics if metric in COLUMN_LOOKUP]

if not available_corr:
    print("⚠️ No hay métricas suficientes para calcular correlaciones.")
else:
    metric_cols = [col for col in offensive_table.columns if any(col.startswith(metric) for metric in available_corr)]

    if len(metric_cols) < 2:
        print("⚠️ No hay suficientes columnas para calcular correlaciones Home/Away.")
    else:
        corr_matrix = offensive_table[metric_cols].corr()

        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, ax=ax)
        ax.set_title("Correlación Home vs Away en métricas ofensivas")
        plt.tight_layout()

        corr_fig_path = FIGURES_DIR / "correlacion_home_away.png"
        fig.savefig(corr_fig_path, dpi=150)
        plt.show()
        print("Figura guardada en", corr_fig_path)

        corr_table_path = TABLES_DIR / "correlacion_home_away.csv"
        corr_matrix.to_csv(corr_table_path)
        print("Matriz de correlación exportada a", corr_table_path)

        display(corr_matrix)

## 6. Conclusiones

- Los splits de Home y Away permiten identificar qué franquicias maximizan el `Win%` en casa y cuáles sostienen su rendimiento a domicilio.
- Las métricas de tiro (`FG%`, `FG3%`, `FT%`) y el diferencial de puntos ayudan a explicar variaciones en la efectividad al cambiar de localización.
- El mapa de correlaciones revela qué indicadores guardan una relación más fuerte con el `Win%` bajo cada escenario, destacando factores como la eficiencia ofensiva y el margen (`PLUS_MINUS`).